In [1]:
# !pip install datasets
# !pip install bert-score
# !pip install rank_bm25

## Imports

In [ ]:
# Standard libraries
import re
import string
import random
import math
from collections import Counter
import time

# PyTorch core
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

# HuggingFace Transformers & Datasets
from transformers import (
    AutoTokenizer,
    AutoModel,
    BertModel,
    get_linear_schedule_with_warmup
)
from datasets import load_dataset, load_from_disk

# Evaluation
from bert_score import score as bert_score

# Ranking
from rank_bm25 import BM25Okapi

# Visualization and summaries
from tqdm import tqdm
from torchsummary import summary

# Colab integration
from google.colab import files


## HotpotQA Dataset (Distractor Setting)

**HotpotQA** is a question-answering dataset that emphasizes multi-hop reasoning, where answering a question requires combining information from multiple documents.

### Dataset Version
- **Configuration**: `distractor` — includes 10 paragraphs per question, with one or more containing supporting facts.

### Split Details
- **Train Set**: 90,447 examples
- **Validation Set**: 7,405 examples

### Features
- `id`: Unique identifier for the example
- `question`: Natural language question
- `answer`: Ground truth answer (string)
- `type`: Type of question (e.g., "comparison", "bridge")
- `level`: Difficulty level (e.g., "easy", "medium", "hard")
- `supporting_facts`: List of (title, sentence_id) pairs that support the answer
- `context`: List of (title, list of sentences) representing documents provided

### Use Case
Ideal for training and evaluating models on:
- Multi-hop reasoning
- Open-domain question answering
- Evidence retrieval


In [4]:
hotpot = load_dataset("hotpot_qa", "distractor", trust_remote_code=True)
print(hotpot)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/9.19k [00:00<?, ?B/s]

hotpot_qa.py:   0%|          | 0.00/6.42k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
        num_rows: 90447
    })
    validation: Dataset({
        features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
        num_rows: 7405
    })
})


In [5]:
train_data = hotpot["train"]
valid_data = hotpot["validation"]

In [6]:
sample = train_data[0]
print(sample["context"])


{'title': ['Radio City (Indian radio station)', 'History of Albanian football', 'Echosmith', "Women's colleges in the Southern United States", 'First Arthur County Courthouse and Jail', "Arthur's Magazine", '2014–15 Ukrainian Hockey Championship', 'First for Women', 'Freeway Complex Fire', 'William Rast'], 'sentences': [["Radio City is India's first private FM radio station and was started on 3 July 2001.", ' It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003).', ' It plays Hindi, English and regional songs.', ' It was launched in Hyderabad in March 2006, in Chennai on 7 July 2006 and in Visakhapatnam October 2007.', ' Radio City recently forayed into New Media in May 2008 with the launch of a music portal - PlanetRadiocity.com that offers music related news, videos, songs, and other music-related features.', ' The Radio station currently plays a mix of Hindi and 

## BERTScore-Based Paragraph Retrieval (Not Used Due to High Compute Cost)

The function `retrieve_top_k_bertscore()` is designed to select the top-k most relevant paragraphs from a HotpotQA context using **BERTScore**, a semantic similarity metric based on pre-trained transformer embeddings.

### What It Does:
- Converts each paragraph in the context into a single string.
- Computes BERTScore (specifically F1) between the question and each paragraph.
- Selects the top-k paragraphs with the highest BERTScore.

### Why We're Not Using It:
Although BERTScore provides a high-quality semantic relevance ranking, it is **computationally expensive**. Even on powerful GPUs like an A100, computing BERTScore for all paragraphs in the distractor setting of HotpotQA is **very slow**, making it impractical for large-scale use during training or real-time inference.

We'll use a faster alternative like **BM25** for paragraph retrieval instead.

In [ ]:
def retrieve_top_k_bertscore(question, context_dict, k=2, model_type="bert-base-uncased"):
    paragraphs = []
    paragraph_texts = []

    for title, sents in zip(context_dict["title"], context_dict["sentences"]):
        para = " ".join(sents)
        paragraphs.append((title, para))
        paragraph_texts.append(para)

    # Compute BERTScore F1 scores
    P, R, F1 = bert_score(paragraph_texts, [question]*len(paragraph_texts), model_type=model_type, verbose=False, device="cuda" if torch.cuda.is_available() else "cpu")
    scores = F1.tolist()

    # Get top-k indices
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [paragraphs[i][1] for i in top_indices]


## BM25-Based Paragraph Retrieval and Tokenization Pipeline

To efficiently train our QA model on the HotpotQA dataset, we use a **BM25-based retrieval** strategy followed by **BERT-style tokenization**. This pipeline segments and encodes each example into a form suitable for span-based QA training.

### Step-by-Step Overview

1. **BM25 Retrieval (`retrieve_top_k_bm25`)**:
   - Joins the sentences in each paragraph.
   - Tokenizes all paragraphs and builds a BM25 index.
   - Ranks paragraphs by relevance to the question and selects the top-k (default: 2).

2. **Segmented Tokenization (`tokenize_segmented`)**:
   - Joins the top-k paragraphs with `[SEP]` separators to form the input context.
   - Tokenizes the question and context with BERT tokenizer (max length: 384).
   - Locates the span of the answer in token space using character offsets.

3. **Preprocessing the Dataset**:
   - Applies `tokenize_segmented` to each example in the training and validation splits.
   - Uses multiprocessing (`num_proc=8`) to speed up preprocessing.
   - Removes original columns and saves only tokenized data with start/end indices.

4. **Caching**:
   - The processed datasets are saved to disk locally:

### Why This Approach?
BM25 offers a **fast and effective** method to narrow down relevant context, which is critical for HotpotQA’s distractor setting. This improves both efficiency and model performance without incurring the computational cost of deep semantic retrieval.



In [ ]:
MAX_LENGTH = 384
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", use_fast=True)

def retrieve_top_k_bm25(question, context_dict, k=2):
    paragraphs = []
    para_texts = []

    for title, sents in zip(context_dict["title"], context_dict["sentences"]):
        para = " ".join(sents)
        paragraphs.append((title, para))
        para_texts.append(para)

    tokenized = [p.lower().split() for p in para_texts]
    bm25 = BM25Okapi(tokenized)
    scores = bm25.get_scores(question.lower().split())

    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [paragraphs[i][1] for i in top_indices]


def tokenize_segmented(example, tokenizer, max_length=384, k=2):
    question = example["question"]
    context_dict = example["context"]
    answer = example["answer"]

    top_paragraphs = retrieve_top_k_bm25(question, context_dict, k=k)
    segmented_context = " [SEP] ".join(top_paragraphs)

    encoded = tokenizer(
        question,
        segmented_context,
        truncation=True,
        max_length=max_length,
        return_offsets_mapping=True,
        return_token_type_ids=True
    )

    input_ids = encoded["input_ids"]
    attention_mask = encoded["attention_mask"]
    token_type_ids = encoded["token_type_ids"]
    offsets = encoded["offset_mapping"]

    context_token_indices = [i for i, ttid in enumerate(token_type_ids) if ttid == 1]

    if not answer or len(answer.strip()) == 0:
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "start_index": -1,
            "end_index": -1
        }

    answer_norm = re.sub(r"\s+", " ", answer.strip().lower())
    context_norm = segmented_context.lower()
    match = re.search(re.escape(answer_norm), context_norm)
    if not match:
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "start_index": -1,
            "end_index": -1
        }

    start_char, end_char = match.start(), match.end()
    start_idx = end_idx = -1
    for idx in context_token_indices:
        sub_start, sub_end = offsets[idx]
        if start_idx == -1 and sub_start >= start_char:
            start_idx = idx
        if sub_end <= end_char:
            end_idx = idx

    if start_idx == -1 or end_idx == -1 or end_idx < start_idx:
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "start_index": -1,
            "end_index": -1
        }

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "start_index": start_idx,
        "end_index": end_idx
    }


# Load HotpotQA
hotpot = load_dataset("hotpot_qa", "distractor", trust_remote_code=True)

# Process train set
train_proc = hotpot["train"].map(
    tokenize_segmented,
    fn_kwargs={"tokenizer": tokenizer, "max_length": MAX_LENGTH, "k": 2},
    remove_columns=hotpot["train"].column_names,
    num_proc=8
)

# Process validation set
val_proc = hotpot["validation"].map(
    tokenize_segmented,
    fn_kwargs={"tokenizer": tokenizer, "max_length": MAX_LENGTH, "k": 2},
    remove_columns=hotpot["validation"].column_names,
    num_proc=8
)

# Save to disk
TRAIN_PATH = r"C:/Users/ashtik/hotpot_cached_train"
VAL_PATH = r"C:/Users/ashtik/hotpot_cached_val"
train_proc.save_to_disk(TRAIN_PATH)
val_proc.save_to_disk(VAL_PATH)

print(f"Done! Saved {len(train_proc)} training and {len(val_proc)} validation samples.")


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map (num_proc=8):   0%|          | 0/90447 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/7405 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/90447 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7405 [00:00<?, ? examples/s]

Done! Saved 90447 training and 7405 validation samples.


## PyTorch Dataset and DataLoader Setup

To train our QA model using PyTorch, we wrap the preprocessed HotpotQA data into a custom `Dataset` and configure efficient `DataLoader`s for both training and validation.

### `HotpotPreprocessedDataset`
A simple wrapper around the Hugging Face `Dataset` object:
- Converts `input_ids` and `attention_mask` into `LongTensor`s.
- Keeps `start_index` and `end_index` for span supervision.

### `collate_fn`
Custom collation function to:
- Pad `input_ids` and `attention_mask` to the maximum sequence length in the batch.
- Stack `start_index` and `end_index` into tensors.
- Enables dynamic batching with proper padding for BERT input format.

### DataLoader Configuration
- **Batch Size**: 256 (adjustable based on available GPU memory).
- **Training**:
  - Shuffled for stochasticity.
  - `pin_memory=True` for faster host-to-device transfer.
- **Validation**:
  - Not shuffled.
  - Uses the same collation logic.

This setup ensures that training is both **efficient** and **compatible with BERT-style input**, while maintaining label alignment for answer span prediction.


In [ ]:


class HotpotPreprocessedDataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return (
            torch.LongTensor(item["input_ids"]),
            torch.LongTensor(item["attention_mask"]),
            item["start_index"],
            item["end_index"]
        )

def collate_fn(batch):
    input_ids_list = [x[0] for x in batch]
    attn_list = [x[1] for x in batch]
    starts = torch.LongTensor([x[2] for x in batch])
    ends = torch.LongTensor([x[3] for x in batch])

    input_ids_padded = pad_sequence(input_ids_list, batch_first=True, padding_value=0)
    attn_padded = pad_sequence(attn_list, batch_first=True, padding_value=0)

    return input_ids_padded, attn_padded, starts, ends


In [10]:

train_dataset = HotpotPreprocessedDataset(load_from_disk(TRAIN_PATH))
val_dataset = HotpotPreprocessedDataset(load_from_disk(VAL_PATH))

batch_size = 256

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          collate_fn=collate_fn, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                        collate_fn=collate_fn, pin_memory=True)


## Neural Turing Machine (NTM) Style Memory Module

This module implements a **differentiable memory** system inspired by the **Neural Turing Machine (NTM)** architecture. It allows a model to read from and write to an external memory bank using content-based addressing.

### Key Features

- **Memory Bank**: A matrix of shape `(batch_size, memory_size, memory_dim)` that stores information over time.
- **Content-Based Addressing**: Reads and writes are guided by cosine similarity between a key vector and memory rows, scaled by a sharpness parameter `beta`.

### Components

1. **`content_address(memory, key, beta)`**:
   - Computes cosine similarity between `key` and each memory row.
   - Applies softmax over scaled similarities to produce a **weight vector** `w`.

2. **`write_to_memory(memory, w, e, a)`**:
   - **Erase Phase**: Erases part of memory using `e` (erase vector) and weight `w`.
   - **Add Phase**: Adds new content using `a` (add vector) and weight `w`.
   - Returns the updated memory.

3. **`forward(memory, interface)`**:
   - Parses the `interface` vector to extract:
     - Write key, write beta, erase vector, add vector
     - Read key, read beta
   - Performs a write, followed by a read.
   - Returns the updated memory and the **read vector**.

### Use Case
This module is particularly useful for **memory-augmented neural networks** like:
- Multi-hop reasoning tasks
- Question answering
- One-shot learning

It helps the model retain and manipulate structured memory across time steps, enabling complex relational reasoning.


In [11]:
class NTM_Memory(nn.Module):
    def __init__(self, memory_size, memory_dim, num_read_heads=1):
        super().__init__()
        self.memory_size = memory_size
        self.memory_dim = memory_dim
        self.num_read_heads = num_read_heads

    def content_address(self, memory, key, beta):
        mem_norm = nn.functional.normalize(memory, dim=-1)
        key_norm = nn.functional.normalize(key, dim=-1)
        sim = torch.bmm(mem_norm, key_norm.unsqueeze(2)).squeeze(-1)
        sim = beta.unsqueeze(1) * sim
        w = torch.softmax(sim, dim=-1)
        return w

    def write_to_memory(self, memory, w, e, a):

        e = torch.sigmoid(e).unsqueeze(1)
        a = a.unsqueeze(1)
        w = w.unsqueeze(-1)

        memory_erase = memory * (1 - w * e)
        memory_add = w * a
        new_mem = memory_erase + memory_add
        return new_mem

    def forward(self, memory, interface):

        bsz = memory.size(0)
        dim = memory.size(2)

        offset = 0
        write_key = interface[:, offset : offset+dim]
        offset += dim
        write_beta = torch.relu(interface[:, offset : offset+1]).squeeze(-1)
        offset += 1
        erase_vec = interface[:, offset : offset+dim]
        offset += dim
        add_vec = interface[:, offset : offset+dim]
        offset += dim
        read_key = interface[:, offset : offset+dim]
        offset += dim
        read_beta = torch.relu(interface[:, offset : offset+1]).squeeze(-1)
        offset += 1

        w_write = self.content_address(memory, write_key, write_beta)
        memory = self.write_to_memory(memory, w_write, erase_vec, add_vec)

        w_read = self.content_address(memory, read_key, read_beta)
        read_vec = torch.bmm(w_read.unsqueeze(1), memory).squeeze(1)  # (b, dim)

        return memory, read_vec


## Advanced Memory-Augmented Neural Network (AdvancedMANN_QA)

This model is a hybrid architecture combining **BERT embeddings**, **RNN-based encoding**, and a **Neural Turing Machine (NTM)-style memory module**, tailored for span-based question answering.

### Architecture Overview

- **BERT Encoder**:
  - Pretrained `bert-base-uncased` is used to embed the input sequence.
  - First two layers are frozen for stability and efficiency.
  - Output: contextual embeddings of shape `(B, L, H)`.

- **LSTM + Memory Controller**:
  - `lstm1` processes BERT embeddings token-by-token.
  - Output is fed into a linear layer to produce the **interface vector** for memory interaction.
  - The **NTM memory module** performs content-based write/read operations at each step.

- **Second LSTM Layer**:
  - Takes concatenated `[lstm1_output, memory_read]` as input.
  - Captures temporal and memory-aware features across the sequence.

- **Projection and QA Head**:
  - All token-wise outputs from the second LSTM are stacked and projected.
  - A linear layer (`qa_head`) produces start and end logits over the sequence.

### Output
- `start_logits`, `end_logits`: Tensors of shape `(B, L)` indicating the probability of each token being the start or end of the answer span.

### Key Benefits
- **Memory Augmentation**: Allows dynamic reasoning over past sequence representations.
- **Hybrid Reasoning**: Combines the strengths of pretrained language models and differentiable memory.
- **Fine-Grained Control**: Token-level recurrence enables more precise span prediction compared to vanilla transformer heads.

This architecture is particularly suited for **multi-hop QA tasks** like HotpotQA, where long-range dependency tracking and external memory access can enhance performance.


In [ ]:


class AdvancedMANN_QA(nn.Module):
    def __init__(self,
                 hidden_dim,
                 memory_size,
                 memory_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.memory_size = memory_size
        self.memory_dim = memory_dim

        self.bert = AutoModel.from_pretrained("bert-base-uncased")
        for name, param in self.bert.named_parameters():
            if "encoder.layer.0" in name or "encoder.layer.1" in name:
                param.requires_grad = False
        bert_dim = self.bert.config.hidden_size

        self.lstm1 = nn.LSTMCell(bert_dim, hidden_dim)

        interface_size = (memory_dim + 1) + memory_dim + memory_dim + (memory_dim + 1)
        self.interface_linear = nn.Linear(hidden_dim, interface_size)
        self.memory_module = NTM_Memory(memory_size, memory_dim, num_read_heads=1)

        self.lstm2 = nn.LSTMCell(hidden_dim + memory_dim, hidden_dim)
        self.proj = nn.Linear(hidden_dim, hidden_dim)

        self.qa_head = nn.Linear(hidden_dim, 2)



    def forward(self, input_ids, attention_mask):
        """
        Returns start_logits, end_logits of shape (B, L).
        """
        bsz, seq_len = input_ids.size()
        device = input_ids.device


        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        embeddings = bert_out.last_hidden_state  # (B, L, H)

        h1 = torch.zeros(bsz, self.hidden_dim, device=device)
        c1 = torch.zeros(bsz, self.hidden_dim, device=device)
        h2 = torch.zeros(bsz, self.hidden_dim, device=device)
        c2 = torch.zeros(bsz, self.hidden_dim, device=device)

        memory = torch.zeros(bsz, self.memory_size, self.memory_dim, device=device)
        hidden_states = []

        for t in range(seq_len):
            mask_t = attention_mask[:, t].float().unsqueeze(-1)
            emb_t = embeddings[:, t]

            h1, c1 = self.lstm1(emb_t, (h1, c1))
            interface_vec = self.interface_linear(h1)
            memory, read_vec = self.memory_module(memory, interface_vec)

            inp2 = torch.cat([h1, read_vec], dim=-1)
            h2, c2 = self.lstm2(inp2, (h2, c2))

            hidden_states.append(h2 * mask_t)

        hidden_stack = torch.stack(hidden_states, dim=0).transpose(0, 1)
        final_summary = h2

        hidden_proj = self.proj(hidden_stack)

        logits = self.qa_head(hidden_proj)  # shape (B, L, 2)
        start_logits, end_logits = logits.split(1, dim=-1)
        start_logits = start_logits.squeeze(-1)  # shape (B, L)
        end_logits = end_logits.squeeze(-1)


        return start_logits, end_logits




## QA Span Prediction Loss

The `qa_span_loss` function computes the training loss for **span-based question answering**, where the model predicts the start and end positions of the answer in the input sequence.

### How It Works

- **Inputs**:
  - `start_logits`, `end_logits`: Model predictions of shape `(B, L)` (batch size × sequence length).
  - `start_positions`, `end_positions`: Ground-truth indices of the answer span, shape `(B,)`.

- **Valid Span Filtering**:
  - Filters out examples where the ground truth span is missing (i.e., index < 0).
  - If no valid spans are present, returns zero loss (with `requires_grad=True` to preserve backprop).

- **Loss Calculation**:
  - Uses `CrossEntropyLoss` over the start and end logits separately.
  - Computes the loss only over valid examples.
  - Returns the **sum of the two losses**.

### Why It’s Used
This is a standard loss function for span-based QA tasks like SQuAD or HotpotQA, where the goal is to predict the exact start and end tokens of the answer within the context. It allows the model to learn precise boundary detection using token-level classification.


In [13]:
def qa_span_loss(start_logits, end_logits, start_positions, end_positions):
    bsz, seq_len = start_logits.size()
    valid_mask = (start_positions >= 0) & (end_positions >= 0)
    if valid_mask.sum() == 0:
        return torch.tensor(0.0, requires_grad=True, device=start_logits.device)

    valid_starts = start_positions[valid_mask]
    valid_ends   = end_positions[valid_mask]
    valid_start_logits = start_logits[valid_mask]
    valid_end_logits   = end_logits[valid_mask]

    loss_fct = nn.CrossEntropyLoss()
    loss_start = loss_fct(valid_start_logits, valid_starts)
    loss_end = loss_fct(valid_end_logits, valid_ends)
    return loss_start + loss_end


In [ ]:
model = AdvancedMANN_QA(
    hidden_dim=128,
    memory_size=32,
    memory_dim=64
)

model = model.cuda()

model = torch.nn.DataParallel(model)


dummy_input = torch.randint(0, tokenizer.vocab_size, (1, 128)).cuda()
dummy_mask = torch.ones(1, 128).cuda()
print(model(dummy_input, dummy_mask))


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

(tensor([[-0.0446, -0.0456, -0.0430, -0.0387, -0.0373, -0.0385, -0.0405, -0.0443,
         -0.0463, -0.0465, -0.0482, -0.0494, -0.0480, -0.0464, -0.0444, -0.0414,
         -0.0407, -0.0372, -0.0354, -0.0339, -0.0352, -0.0369, -0.0391, -0.0408,
         -0.0410, -0.0416, -0.0430, -0.0450, -0.0434, -0.0442, -0.0441, -0.0437,
         -0.0415, -0.0375, -0.0379, -0.0404, -0.0412, -0.0419, -0.0453, -0.0482,
         -0.0473, -0.0462, -0.0449, -0.0449, -0.0431, -0.0457, -0.0470, -0.0468,
         -0.0422, -0.0375, -0.0338, -0.0335, -0.0313, -0.0307, -0.0292, -0.0256,
         -0.0254, -0.0242, -0.0245, -0.0245, -0.0236, -0.0239, -0.0224, -0.0223,
         -0.0221, -0.0227, -0.0210, -0.0212, -0.0236, -0.0255, -0.0320, -0.0292,
         -0.0295, -0.0299, -0.0321, -0.0332, -0.0348, -0.0338, -0.0359, -0.0376,
         -0.0373, -0.0374, -0.0384, -0.0378, -0.0362, -0.0358, -0.0341, -0.0338,
         -0.0310, -0.0299, -0.0324, -0.0349, -0.0376, -0.0378, -0.0406, -0.0399,
         -0.0379, -0.0344, 

### Training Loop: `train_one_epoch`

- **AMP (Autocast + GradScaler)**:
  - Uses mixed precision to reduce memory usage and speed up training on GPUs.
  - Scales loss before backward pass and updates the optimizer safely.

- **Loss + Accuracy Calculation**:
  - Computes span loss using `qa_span_loss`.
  - Calculates accuracy based on whether both start and end positions match ground truth.
  - Tracks average loss and accuracy across valid samples.

- **Scheduler Step**:
  - Updates the learning rate using a scheduler after every optimizer step.

In [ ]:

# model = torch.compile(model, mode="reduce-overhead")

# Optimizer + AMP scaler
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scaler = GradScaler()

# Training loop
def train_one_epoch(model, loader, optimizer, scheduler, scaler):
    model.train()
    total_loss = 0.0
    total_count = 0
    total_correct = 0

    prog_bar = tqdm(loader, desc="Training", leave=False)

    for batch in prog_bar:
        input_ids, attn_mask, starts, ends = batch
        input_ids = input_ids.cuda()
        attn_mask = attn_mask.cuda()
        starts = starts.cuda()
        ends = ends.cuda()

        optimizer.zero_grad()

        with autocast(device_type="cuda"):
            start_logits, end_logits = model(input_ids, attn_mask)
            loss = qa_span_loss(start_logits, end_logits, starts, ends)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        valid_mask = (starts >= 0)
        pred_start = start_logits.argmax(dim=-1)
        pred_end = end_logits.argmax(dim=-1)
        correct = ((pred_start == starts) & (pred_end == ends) & valid_mask).sum().item()
        total_valid = valid_mask.sum().item()

        batch_acc = correct / max(total_valid, 1)
        prog_bar.set_postfix(loss=loss.item(), acc=batch_acc)

        count = max(total_valid, 1)
        total_loss += loss.item() * count
        total_count += count
        total_correct += correct

    avg_loss = total_loss / total_count
    avg_acc = total_correct / total_count
    return avg_loss, avg_acc


def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_count = 0
    correct_spans = 0
    total_spans = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc="Validating", leave=False):
            input_ids, attn_mask, starts, ends = batch
            input_ids = input_ids.cuda()
            attn_mask = attn_mask.cuda()
            starts = starts.cuda()
            ends = ends.cuda()

            start_logits, end_logits = model(input_ids, attn_mask)
            loss = qa_span_loss(start_logits, end_logits, starts, ends)

            count = (starts >= 0).sum().item()
            total_loss += loss.item() * max(count, 1)
            total_count += max(count, 1)

            valid_mask = (starts >= 0) & (ends >= 0)
            pred_start = start_logits.argmax(dim=-1)
            pred_end   = end_logits.argmax(dim=-1)

            valid_indices = torch.where(valid_mask)[0]
            for i in valid_indices:
                total_spans += 1
                if pred_start[i] == starts[i] and pred_end[i] == ends[i]:
                    correct_spans += 1

    avg_loss = total_loss / (total_count if total_count > 0 else 1)
    span_acc = correct_spans / (total_spans if total_spans > 0 else 1)
    return avg_loss, span_acc

In [17]:
train_data = load_from_disk(TRAIN_PATH)
valid_spans = train_data.filter(lambda x: x["start_index"] >= 0 and x["end_index"] >= 0)
print(f"Kept {len(valid_spans)} / {len(train_data)} valid training examples")



Filter:   0%|          | 0/90447 [00:00<?, ? examples/s]

Kept 54505 / 90447 valid training examples


## Training Configuration and Loop

This section sets up the final training loop for the **AdvancedMANN_QA** model using a two-tier learning rate strategy and learning rate scheduling.


### Optimizer: `AdamW`
We use `AdamW` with two separate parameter groups:
- **BERT Parameters**:
  - Learning Rate: `3e-5`
  - Slower updates to preserve pretrained knowledge.
- **Non-BERT Parameters** (LSTM, memory, QA head, etc.):
  - Learning Rate: `1e-3`
  - Faster learning for newly initialized components.


### Scheduler: `Linear Schedule with Warmup`
- Gradually warms up the learning rate for the first 10% of total steps.
- Then linearly decays the learning rate.
- Helps stabilize training in the early epochs.


### Mixed Precision Support
- Uses `GradScaler` for safe scaling of the loss during mixed precision training.
- Reduces GPU memory usage and speeds up training on modern hardware (e.g., A100).


### Training Loop
- Runs for `EPOCHS = 5`.
- Tracks training and validation loss/accuracy per epoch.
- Prints epoch summary with runtime and performance metrics.


In [ ]:


EPOCHS = 5

bert_params = list(model.module.bert.parameters()) if isinstance(model, torch.nn.DataParallel) else list(model.bert.parameters())
other_params = [p for n, p in model.named_parameters() if "bert" not in n]

optimizer = optim.AdamW([
    {"params": bert_params, "lr": 3e-5},
    {"params": other_params, "lr": 1e-3}
])

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

scaler = GradScaler()

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}")
    start_time = time.time()

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, scheduler, scaler)
    val_loss, val_acc = evaluate(model, val_loader)

    elapsed = time.time() - start_time
    print(f"Done in {elapsed:.2f}s | Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")





Epoch 1


Done in 922.87s | Train Loss: 3.6672 | Acc: 0.3641 | Val Acc: 0.5424

Epoch 2


Done in 924.03s | Train Loss: 1.1428 | Acc: 0.6697 | Val Acc: 0.5729

Epoch 3


Done in 925.81s | Train Loss: 0.8738 | Acc: 0.7369 | Val Acc: 0.5896

Epoch 4


Done in 926.46s | Train Loss: 0.7103 | Acc: 0.7793 | Val Acc: 0.5948

Epoch 5


Done in 925.53s | Train Loss: 0.5924 | Acc: 0.8131 | Val Acc: 0.5981


## Prediction & Inference Utilities

This section defines utility functions for making predictions with the trained QA model and testing it on custom examples.

### `predict_span(model, input_ids, attention_mask)`
- Runs the model in evaluation mode.
- Returns the predicted start and end token indices for the answer span.

### `decode_span(input_ids, start_idx, end_idx)`
- Converts token indices back to readable text using the tokenizer.
- Handles invalid span predictions gracefully.

### Example Prediction on Validation Set
- Selects the first validation sample.
- Runs inference and prints both the **predicted** and **true** spans.
  
### `prepare_input(question, context_paragraphs, tokenizer)`
- Converts a list of `(title, sentences)` pairs into a flat string.
- Tokenizes the combined question + context for model input.
### `decode_answer(input_ids, start_idx, end_idx, tokenizer)`
- Decodes a predicted span into readable text.
- Handles invalid cases like inverted or out-of-bounds spans.

### `ask_question(model, tokenizer, question, context_paragraphs)`
- Wraps everything into a single function:
  - Prepares the input.
  - Runs the model.
  - Decodes the final answer.
- Returns both the predicted answer and its token span.


In [ ]:
def predict_span(model, input_ids, attention_mask):
    model.eval()
    with torch.no_grad():
        start_logits, end_logits = model(input_ids, attention_mask)
        start_idx = start_logits.argmax(dim=-1).item()
        end_idx = end_logits.argmax(dim=-1).item()
    return start_idx, end_idx

def decode_span(input_ids, start_idx, end_idx):
    tokens = tokenizer.convert_ids_to_tokens(input_ids.cpu().tolist())

    if start_idx < 0 or end_idx < 0 or end_idx < start_idx or end_idx >= len(tokens):
        return "[Invalid span]"

    span_tokens = tokens[start_idx:end_idx + 1]
    return tokenizer.convert_tokens_to_string(span_tokens)



example = val_dataset[0]
ids = torch.LongTensor(example[0]).unsqueeze(0).cuda()
attn = torch.LongTensor(example[1]).unsqueeze(0).cuda()
true_start, true_end = example[2], example[3]

pred_s, pred_e = predict_span(model, ids, attn)
pred_answer = decode_span(ids[0], pred_s, pred_e)

print("Predicted span:", (pred_s, pred_e), "->", pred_answer)
print("True span:", (true_start, true_end))


Predicted span: (203, 269) -> north little rock – conway metropolitan statistical area. woodson and its accompanying woodson lake and wood hollow are the namesake for ed wood sr., a prominent plantation owner, trader, and businessman at the turn of the 20th century. woodson is adjacent to the wood plantation, the largest of the plantations own by ed wood sr. [SEP]
True span: (-1, -1)


In [ ]:


def prepare_input(question, context_paragraphs, tokenizer, max_length=256):
    titles = [x[0] for x in context_paragraphs]
    sentences = [x[1] for x in context_paragraphs]

    context_sents = [" ".join(s) for s in sentences]
    context_text = " ".join(context_sents)

    encoded = tokenizer(
        question,
        context_text,
        truncation=True,
        max_length=max_length,
        return_offsets_mapping=False,
        return_token_type_ids=True
    )

    return torch.tensor(encoded["input_ids"]).unsqueeze(0), torch.tensor(encoded["attention_mask"]).unsqueeze(0)

def decode_answer(input_ids, start_idx, end_idx, tokenizer):
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0].cpu().tolist())
    if start_idx > end_idx or end_idx >= len(tokens):
        return "[Invalid prediction]"
    return tokenizer.convert_tokens_to_string(tokens[start_idx:end_idx+1])

def ask_question(model, tokenizer, question, context_paragraphs):
    input_ids, attention_mask = prepare_input(question, context_paragraphs, tokenizer)
    input_ids = input_ids.cuda()
    attention_mask = attention_mask.cuda()

    model.eval()
    with torch.no_grad():
        start_logits, end_logits = model(input_ids, attention_mask)
        start_idx = start_logits.argmax(dim=-1).item()
        end_idx = end_logits.argmax(dim=-1).item()

    answer = decode_answer(input_ids, start_idx, end_idx, tokenizer)
    return answer, (start_idx, end_idx)

question = "Who owns Radio City FM?"
context_paragraphs = [
    ["Radio City (Indian radio station)", [
        "Radio City is India's first private FM radio station and was started on 3 July 2001.",
        "It broadcasts on 91.1 megahertz from Mumbai, Bengaluru, Lucknow and New Delhi.",
        "Abraham Thomas is the CEO of the company.",
        "Radio City was acquired by Music Broadcast Ltd."
    ]]
]

answer, span = ask_question(model, tokenizer, question, context_paragraphs)

print("Question:", question)
print("Context:", " ".join(context_paragraphs[0][1]))
print("Predicted Span:", span)
print("Answer:", answer)


Question: Who owns Radio City FM?
Context: Radio City is India's first private FM radio station and was started on 3 July 2001. It broadcasts on 91.1 megahertz from Mumbai, Bengaluru, Lucknow and New Delhi. Abraham Thomas is the CEO of the company. Radio City was acquired by Music Broadcast Ltd.
Predicted Span: (61, 65)
Answer: music broadcast ltd. [SEP]


In [ ]:
save_path = "mann_qa_model.pt"
torch.save(model.module.state_dict(), save_path) 

files.download(save_path)  
print(f"Model saved to {save_path}")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Model saved to mann_qa_model.pt


## Final Evaluation: Exact Match and F1 Metrics

To assess the model’s performance more thoroughly, we evaluate it using two standard QA metrics: **Exact Match (EM)** and **F1 Score**, as used in benchmarks like SQuAD and HotpotQA.

---

### Normalization (`normalize_answer`)
Applies a series of transformations to standardize answers before comparison:
- Lowercasing
- Removing punctuation
- Removing articles ("a", "an", "the")
- Fixing extra whitespace

This ensures surface-level differences (e.g., "The UN" vs. "UN") do not penalize the model.

---

### Metrics

- **Exact Match (EM)**:
  - Checks if the normalized prediction is **exactly equal** to the normalized ground truth.

- **F1 Score**:
  - Measures the **token-level overlap** between predicted and gold answers.
  - Balances **precision** and **recall**:
    - Precision = predicted tokens that are correct
    - Recall = gold tokens that were predicted
    - F1 = harmonic mean of precision and recall

---

### Evaluation Function: `final_eval(model, dataset, tokenizer, max_samples=1000)`

- Runs inference on up to `max_samples` from the dataset.
- Decodes both predicted and true spans into text.
- Computes EM and F1 for each example.
- Prints and returns the **average EM and F1 scores**.

---

In [ ]:


def normalize_answer(s):
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)
    def white_space_fix(text):
        return ' '.join(text.split())
    def remove_punc(text):
        return ''.join(ch for ch in text if ch not in set(string.punctuation))
    def lower(text):
        return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))

def compute_exact(a_gold, a_pred):
    return int(normalize_answer(a_gold) == normalize_answer(a_pred))

def compute_f1(a_gold, a_pred):
    gold_toks = normalize_answer(a_gold).split()
    pred_toks = normalize_answer(a_pred).split()
    common = Counter(gold_toks) & Counter(pred_toks)
    num_same = sum(common.values())
    if len(gold_toks) == 0 or len(pred_toks) == 0:
        return int(gold_toks == pred_toks)
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_toks)
    recall = num_same / len(gold_toks)
    return 2 * precision * recall / (precision + recall)

def final_eval(model, dataset, tokenizer, max_samples=1000):
    model.eval()
    total_em, total_f1 = 0.0, 0.0
    count = 0

    print("Running Final Evaluation")
    for ex in tqdm(dataset.select(range(min(len(dataset), max_samples))), desc="Evaluating"):
        input_ids = torch.LongTensor(ex["input_ids"]).unsqueeze(0).cuda()
        attn_mask = torch.LongTensor(ex["attention_mask"]).unsqueeze(0).cuda()
        true_start, true_end = ex["start_index"], ex["end_index"]

        pred_start, pred_end = predict_span(model, input_ids, attn_mask)
        pred_ans = decode_span(input_ids[0], pred_start, pred_end)
        gold_ans = decode_span(input_ids[0], true_start, true_end)

        total_em += compute_exact(gold_ans, pred_ans)
        total_f1 += compute_f1(gold_ans, pred_ans)
        count += 1

    avg_em = total_em / count
    avg_f1 = total_f1 / count
    print(f"Final Eval on {count} samples | EM: {avg_em:.4f} | F1: {avg_f1:.4f}")
    return avg_em, avg_f1


In [ ]:

# Load original preprocessed val data
val_hf = load_from_disk(VAL_PATH)

# Filter valid spans
val_valid = val_hf.filter(lambda x: x["start_index"] >= 0 and x["end_index"] >= 0)

# Then evaluate on that
final_eval(model, val_valid, tokenizer, max_samples=1000)


Filter:   0%|          | 0/7405 [00:00<?, ? examples/s]

Running Final Evaluation


Evaluating: 100%|██████████| 1000/1000 [04:23<00:00,  3.80it/s]

Final Eval on 1000 samples | EM: 0.6250 | F1: 0.9050


(0.625, 0.9050418189324926)

### Metrics

#### EM: 62.5%, F1 (overlap): 90.5%

## Manual Evaluation Demo

In [24]:
def run_manual_eval(model, tokenizer):
    test_cases = [
        {
            "question": "Who is the founder of Microsoft?",
            "context_paragraphs": [
                ["Microsoft", [
                    "Microsoft Corporation is a multinational technology company.",
                    "It was founded by Bill Gates and Paul Allen in 1975.",
                    "Microsoft is headquartered in Redmond, Washington."
                ]]
            ]
        },
        {
            "question": "When did the Berlin Wall fall?",
            "context_paragraphs": [
                ["Berlin Wall", [
                    "The Berlin Wall was a guarded concrete barrier that physically and ideologically divided Berlin from 1961 to 1989.",
                    "The Wall fell on November 9, 1989, marking the beginning of German reunification.",
                    "Its demolition officially started in June 1990 and ended in 1992."
                ]]
            ]
        },
        {
            "question": "Where is the headquarters of the company led by Elon Musk?",
            "context_paragraphs": [
                ["Tesla, Inc.", [
                    "Tesla is an American electric vehicle and clean energy company.",
                    "Its CEO is Elon Musk.",
                    "Tesla is headquartered in Palo Alto, California."
                ]]
            ]
        },
        {
            "question": "Who won the 2008 US Presidential election?",
            "context_paragraphs": [
                ["US Elections", [
                    "The 2004 US election was won by George W. Bush.",
                    "In 2012, Barack Obama was re-elected.",
                    "The 2008 United States presidential election was won by Barack Obama, defeating John McCain."
                ]]
            ]
        },
        {
            "question": "Who directed the movie Inception?",
            "context_paragraphs": [
                ["Inception", [
                    "Inception is a 2010 science fiction film.",
                    "It was written and directed by Christopher Nolan.",
                    "The film stars Leonardo DiCaprio as a professional thief."
                ]]
            ]
        },
        {
            "question": "Who owns Radio City FM?",
            "context_paragraphs": [
                ["Radio City (Indian radio station)", [
                    "Radio City is India's first private FM radio station and was started on 3 July 2001.",
                    "It broadcasts on 91.1 megahertz from Mumbai, Bengaluru, Lucknow and New Delhi.",
                    "Abraham Thomas is the CEO of the company.",
                    "Radio City was acquired by Music Broadcast Ltd."
                ]]
            ]
        }
    ]

    for i, ex in enumerate(test_cases, 1):
        question = ex["question"]
        context_paragraphs = ex["context_paragraphs"]

        answer, span = ask_question(model, tokenizer, question, context_paragraphs)

        print(f"\n Example {i}")
        print("Question:", question)
        print("Context:", " ".join(context_paragraphs[0][1]))
        print("Predicted Span:", span)
        print("Answer:", answer)


In [25]:
run_manual_eval(model, tokenizer)



 Example 1
Question: Who is the founder of Microsoft?
Context: Microsoft Corporation is a multinational technology company. It was founded by Bill Gates and Paul Allen in 1975. Microsoft is headquartered in Redmond, Washington.
Predicted Span: (21, 38)
Answer: bill gates and paul allen in 1975. microsoft is headquartered in redmond, washington. [SEP]

 Example 2
Question: When did the Berlin Wall fall?
Context: The Berlin Wall was a guarded concrete barrier that physically and ideologically divided Berlin from 1961 to 1989. The Wall fell on November 9, 1989, marking the beginning of German reunification. Its demolition officially started in June 1990 and ended in 1992.
Predicted Span: (33, 57)
Answer: november 9, 1989, marking the beginning of german reunification. its demolition officially started in june 1990 and ended in 1992. [SEP]

 Example 3
Question: Where is the headquarters of the company led by Elon Musk?
Context: Tesla is an American electric vehicle and clean energy compan

## References

### Datasets & Benchmarks
- **HotpotQA**  
  Yang, Y., Qi, P., Zhang, S., Bengio, Y., Cohen, W., Salakhutdinov, R., & Manning, C. D. (2018).  
  *HotpotQA: A Dataset for Diverse, Explainable Multi-hop Question Answering*. EMNLP 2018.  
  [Paper](https://arxiv.org/abs/1809.09600) | [Hugging Face Dataset](https://huggingface.co/datasets/hotpot_qa)

---

### Memory-Augmented Models
- **Neural Turing Machines (NTM)**  
  Graves, A., Wayne, G., & Danihelka, I. (2014).  
  *Neural Turing Machines*. arXiv preprint.  
  [Paper](https://arxiv.org/abs/1410.5401)

- **Memory Networks**  
  Weston, J., Chopra, S., & Bordes, A. (2014).  
  *Memory Networks*. arXiv preprint.  
  [Paper](https://arxiv.org/abs/1410.3916)

---

### Model Components
- **BERT**  
  Devlin, J., Chang, M.-W., Lee, K., & Toutanova, K. (2019).  
  *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. NAACL 2019.  
  [Paper](https://arxiv.org/abs/1810.04805) | [Model](https://huggingface.co/bert-base-uncased)

- **Transformers Library**  
  Wolf, T., Debut, L., Sanh, V., Chaumond, J., Delangue, C., Moi, A., ... & Rush, A. M. (2020).  
  *Transformers: State-of-the-Art Natural Language Processing*. EMNLP 2020.  
  [Paper](https://arxiv.org/abs/1910.03771) | [GitHub](https://github.com/huggingface/transformers)

---

### Evaluation Metrics
- **SQuAD Evaluation (EM, F1)**  
  Based on official metric definitions used in the [SQuAD leaderboard](https://rajpurkar.github.io/SQuAD-explorer/).  
  Normalization and scoring functions are adapted from the [official SQuAD evaluation script](https://github.com/allenai/bi-att-flow/blob/master/squad/evaluate-v1.1.py).

---

### Retrieval Methods
- **BM25**  
  Robertson, S., & Zaragoza, H. (2009).  
  *The Probabilistic Relevance Framework: BM25 and Beyond*.  
  [Wikipedia Overview](https://en.wikipedia.org/wiki/Okapi_BM25)

- **BERTScore**  
  Zhang, T., Kishore, V., Wu, F., Weinberger, K. Q., & Artzi, Y. (2020).  
  *BERTScore: Evaluating Text Generation with BERT*. ICLR 2020.  
  [Paper](https://arxiv.org/abs/1904.09675) | [GitHub](https://github.com/Tiiiger/bert_score)
